# Tahap 5: Model Evaluation
**SubCPMK-3 — Penalaran Komputer | NIM: 202310370311358**

Tujuan: Mengukur dan menganalisis performa sistem CBR (retrieval & prediksi) menggunakan Accuracy, Precision, Recall, dan F1-score.

In [ ]:
import json
import pickle
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from sklearn.metrics.pairwise import cosine_similarity
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

PROCESSED_DIR = Path('../data/processed')
EVAL_DIR      = Path('../data/eval')
RESULTS_DIR   = Path('../data/results')

# Load model & data
with open(PROCESSED_DIR / 'tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)
with open(PROCESSED_DIR / 'svm_model.pkl', 'rb') as f:
    svm = pickle.load(f)
with open(PROCESSED_DIR / 'label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

df_clean  = pd.read_csv(PROCESSED_DIR / 'cases_clean.csv')
df_index  = pd.read_csv(PROCESSED_DIR / 'case_index.csv')
X_all     = np.load(PROCESSED_DIR / 'tfidf_matrix.npy')
df_pred   = pd.read_csv(RESULTS_DIR / 'predictions.csv')

# Hapus baris unknown
df_model = df_clean[df_clean['label'] != 'unknown'].copy()
print(f'Dataset evaluasi: {len(df_model)} kasus')

## 5.1 Evaluasi Retrieval: Top-k Accuracy

In [ ]:
STOPWORDS_ID = set(stopwords.words('indonesian'))
all_ids    = df_index['case_id'].values
all_labels = df_index['label'].values

def preprocess_query(query):
    query = query.lower()
    query = re.sub(r'[^a-zA-Z\s]', ' ', query)
    return ' '.join([t for t in query.split() if t not in STOPWORDS_ID and len(t) > 2])

def retrieve(query, k=5):
    q_vec = tfidf.transform([preprocess_query(query)]).toarray()
    sims  = cosine_similarity(q_vec, X_all).flatten()
    top_k = np.argsort(sims)[::-1][:k]
    return [(all_ids[i], all_labels[i], float(sims[i])) for i in top_k]

# Load queries evaluasi
with open(EVAL_DIR / 'queries.json', encoding='utf-8') as f:
    eval_queries = json.load(f)

# Evaluasi retrieval: apakah ground truth ada di top-k?
results_retrieval = []
for k_val in [1, 3, 5]:
    hits = 0
    label_preds, label_truths = [], []

    for q in eval_queries:
        top_k = retrieve(q['query_text'], k=k_val)
        retrieved_ids    = [r[0] for r in top_k]
        retrieved_labels = [r[1] for r in top_k]

        # Hit@k: apakah ground_truth ada di retrieved_ids?
        if q['ground_truth_id'] in retrieved_ids:
            hits += 1

        # Untuk klasifikasi: ambil label Top-1
        if k_val == 1:
            label_preds.append(retrieved_labels[0])
            label_truths.append(q['ground_truth_label'])

    hit_rate = hits / len(eval_queries) if eval_queries else 0
    results_retrieval.append({'k': k_val, 'hit_rate': hit_rate})
    print(f'Hit@{k_val}: {hit_rate:.4f} ({hits}/{len(eval_queries)})')

df_hit = pd.DataFrame(results_retrieval)
print()

## 5.2 Evaluasi Klasifikasi SVM

In [ ]:
from sklearn.model_selection import train_test_split

X = tfidf.transform(df_model['text_clean'].fillna(''))
y = le.transform(df_model['label'])

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y if len(set(y)) > 1 else None
)

y_pred = svm.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print('=== Evaluasi SVM (Test Set) ===')
print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-Score  : {f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

## 5.3 Evaluasi Prediksi Solusi (predictions.csv)

In [ ]:
def eval_retrieval(queries_df, ground_truth_col='ground_truth_label',
                   pred_col='predicted_label', k=5):
    """Hitung metrics Accuracy, Precision, Recall, F1 dari hasil prediksi."""
    valid = queries_df.dropna(subset=[ground_truth_col, pred_col])
    valid = valid[
        (valid[ground_truth_col] != 'unknown') &
        (valid[pred_col] != 'unknown')
    ]
    if valid.empty:
        print('Tidak ada data valid untuk dievaluasi.')
        return {}

    y_true = valid[ground_truth_col]
    y_pred = valid[pred_col]

    return {
        'n_samples' : len(valid),
        'accuracy'  : accuracy_score(y_true, y_pred),
        'precision' : precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall'    : recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1'        : f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }

metrics_pred = eval_retrieval(df_pred)
print('=== Evaluasi Prediksi Solusi ===')
for k, v in metrics_pred.items():
    print(f'{k:12s}: {v:.4f}' if isinstance(v, float) else f'{k:12s}: {v}')

## 5.4 Visualisasi & Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Bar chart performa SVM ---
metrics_svm = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1}
ax1 = axes[0]
bars = ax1.bar(metrics_svm.keys(), metrics_svm.values(),
               color=['#4C8BE8', '#52C98C', '#E87B4C', '#9B5FDE'])
ax1.set_title('Performa Model SVM (TF-IDF)', fontsize=13, fontweight='bold')
ax1.set_ylim(0, 1.1)
ax1.set_ylabel('Score')
for bar, val in zip(bars, metrics_svm.values()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', fontsize=11)

# --- Plot 2: Hit Rate ---
ax2 = axes[1]
ax2.bar([f'Hit@{r["k"]}' for r in results_retrieval],
        [r['hit_rate'] for r in results_retrieval],
        color=['#E8D44C', '#4CC4E8', '#E84C7B'])
ax2.set_title('Hit Rate Retrieval (Top-k)', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 1.1)
ax2.set_ylabel('Hit Rate')
for i, r in enumerate(results_retrieval):
    ax2.text(i, r['hit_rate'] + 0.02, f"{r['hit_rate']:.3f}", ha='center', fontsize=11)

plt.tight_layout()
plt.savefig(EVAL_DIR / 'performance_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: data/eval/performance_chart.png')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — SVM Classifier', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig(EVAL_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: data/eval/confusion_matrix.png')

## 5.5 Analisis Kegagalan (Error Analysis)

In [ ]:
# Identifikasi kasus yang salah diprediksi
df_errors = df_pred[
    (df_pred['ground_truth_label'] != 'unknown') &
    (df_pred['predicted_label'] != 'unknown') &
    (df_pred['ground_truth_label'] != df_pred['predicted_label'])
].copy()

print(f'Kasus gagal diprediksi: {len(df_errors)} dari {len(df_pred)}')

if not df_errors.empty:
    print('\nDetail kasus gagal:')
    display(df_errors[['query_id', 'ground_truth_label', 'predicted_label', 'top_5_case_ids']])

    print('\nAnalisis Penyebab Kegagalan:')
    print('1. Kasus dengan kata kunci ambigu (pasal yang overlap antar kelas)')
    print('2. Jumlah sampel per kelas tidak seimbang (imbalanced classes)')
    print('3. Ringkasan query terlalu pendek → vektor TF-IDF tidak representatif')

    print('\nRekomendasi Perbaikan:')
    print('- Tambah volume data (>50 dokumen) untuk kelas minoritas')
    print('- Gunakan SMOTE untuk oversampling kelas minoritas')
    print('- Coba IndoBERT embedding untuk representasi semantik yang lebih kaya')
    print('- Tuning hyperparameter SVM (C, kernel)')

## 5.6 Simpan Metrics ke CSV

In [ ]:
# Retrieval metrics
df_retrieval_metrics = pd.DataFrame([
    {'model': 'TF-IDF + SVM', 'k': 1, 'metric': 'Hit Rate', 'value': results_retrieval[0]['hit_rate']},
    {'model': 'TF-IDF + SVM', 'k': 3, 'metric': 'Hit Rate', 'value': results_retrieval[1]['hit_rate']},
    {'model': 'TF-IDF + SVM', 'k': 5, 'metric': 'Hit Rate', 'value': results_retrieval[2]['hit_rate']},
    {'model': 'TF-IDF + SVM', 'k': '-', 'metric': 'Accuracy',  'value': acc},
    {'model': 'TF-IDF + SVM', 'k': '-', 'metric': 'Precision', 'value': prec},
    {'model': 'TF-IDF + SVM', 'k': '-', 'metric': 'Recall',    'value': rec},
    {'model': 'TF-IDF + SVM', 'k': '-', 'metric': 'F1-Score',  'value': f1},
])
df_retrieval_metrics.to_csv(EVAL_DIR / 'retrieval_metrics.csv', index=False)

# Prediction metrics
df_prediction_metrics = pd.DataFrame([metrics_pred])
df_prediction_metrics.to_csv(EVAL_DIR / 'prediction_metrics.csv', index=False)

print('Saved: data/eval/retrieval_metrics.csv')
print('Saved: data/eval/prediction_metrics.csv')
display(df_retrieval_metrics)

print('\n>>> Tahap 5 SELESAI. Proyek CBR lengkap!')
print('Upload repository ke GitHub dan submit link ke LMS.')